# IMDB sentiment classification with frozen e5-small-v2 embeddings

This notebook builds a binary sentiment classifier that exists in order to be
explained. The next notebook applies SHAP to it, so the real deliverable here is
`predict_probabilities`, a function mapping raw review text to class
probabilities, plus cached artifacts so the SHAP notebook starts immediately.

The architecture is deliberately simple. A frozen `intfloat/e5-small-v2` encoder
turns each review into a 384 dimensional vector, and a small multilayer
perceptron classifies that vector. Nothing is backpropagated through the
transformer, so the whole thing trains on a CPU in seconds once the embeddings
are cached.

Work runs on a stratified sample of 2,000 reviews rather than the full 50,000.
Encoding the full set takes well over 40 minutes on a CPU, which is too slow to
iterate against, and SHAP is the point here rather than the last percentage
point of accuracy. Embeddings are checkpointed to disk under a directory named
after the sample size, so raising `SAMPLE_SIZE` later creates a separate
checkpoint instead of silently reusing vectors from a different sample.

Two decisions are driven by the SHAP step rather than by accuracy:

1. **Reviews longer than the encoder window are dropped, not truncated.**
   Truncation would hand SHAP tokens the model never reads. Those tokens receive
   near zero attribution, which reads as "the model ignored the ending" when the
   truth is "the model never received the ending". Dropping guarantees every
   token shown to SHAP genuinely influences the output.
2. **The e5 `query: ` prefix is added inside `predict_probabilities`, never
   stored in the text column.** SHAP therefore masks only the review itself and
   never perturbs a fixed instruction string.

## Configuration

In [1]:
from pathlib import Path

EMBEDDING_MODEL_NAME = "intfloat/e5-small-v2"
MAXIMUM_TOKEN_COUNT = 512
QUERY_PREFIX = "query: "
RANDOM_SEED = 42
SAMPLE_SIZE = 2000
TEST_FRACTION = 0.2
BATCH_SIZE = 64
TORCH_THREAD_COUNT = 8

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIRECTORY = PROJECT_ROOT / "data"
CHECKPOINT_DIRECTORY = PROJECT_ROOT / "artifacts" / f"sample_{SAMPLE_SIZE}"

DATA_DIRECTORY.mkdir(exist_ok=True)
CHECKPOINT_DIRECTORY.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Checkpoint directory: {CHECKPOINT_DIRECTORY}")

Project root: /home/user/projects/shapley
Checkpoint directory: /home/user/projects/shapley/artifacts/sample_2000


## Download the dataset

The Kaggle download endpoint redirects to a signed storage URL and needs no
Kaggle account or API token. The step is idempotent, so rerunning the notebook
does not redownload.

In [2]:
import urllib.request
import zipfile

KAGGLE_DOWNLOAD_URL = (
    "https://www.kaggle.com/api/v1/datasets/download/"
    "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
)
CSV_PATH = DATA_DIRECTORY / "IMDB Dataset.csv"

if not CSV_PATH.exists():
    archive_path = DATA_DIRECTORY / "imdb.zip"
    print("Downloading the IMDB dataset from Kaggle")
    urllib.request.urlretrieve(KAGGLE_DOWNLOAD_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIRECTORY)
    archive_path.unlink()

print(f"Dataset ready at {CSV_PATH}")

Dataset ready at /home/user/projects/shapley/data/IMDB Dataset.csv


## Clean the HTML, then deduplicate

29,200 of the 50,000 reviews contain a literal `<br />` tag. Left in place it
becomes a top ranked SHAP feature and the explanations are worthless.

Cleaning runs *before* deduplication on purpose, so that reviews differing only
in markup collapse into one another. The assertions make a corrupted or partial
download fail loudly instead of quietly training on noise.

In [3]:
import re

import pandas as pd

HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
REPEATED_WHITESPACE_PATTERN = re.compile(r"\s+")


def remove_html(text: str) -> str:
    """Strip HTML markup and collapse the whitespace it leaves behind."""
    without_tags = HTML_TAG_PATTERN.sub(" ", text)
    return REPEATED_WHITESPACE_PATTERN.sub(" ", without_tags).strip()


raw_reviews = pd.read_csv(CSV_PATH)
assert len(raw_reviews) == 50_000, f"Expected 50000 rows, found {len(raw_reviews)}"
assert raw_reviews["sentiment"].value_counts().to_dict() == {
    "positive": 25_000,
    "negative": 25_000,
}, "Class balance is not the expected 25000 positive and 25000 negative"

reviews = pd.DataFrame(
    {
        "text": raw_reviews["review"].map(remove_html),
        "label": (raw_reviews["sentiment"] == "positive").astype(int),
    }
).drop_duplicates(subset="text").reset_index(drop=True)

print(f"Rows before deduplication: {len(raw_reviews)}")
print(f"Unique reviews after cleaning HTML: {len(reviews)}")
print(f"Class balance: {reviews['label'].value_counts().to_dict()}")

Rows before deduplication: 50000
Unique reviews after cleaning HTML: 49578
Class balance: {1: 24882, 0: 24696}


## Drop reviews that do not fit the encoder window

Fit is decided by tokenizing rather than by estimating a words to tokens ratio,
which accounts for the two special tokens and the query prefix automatically.

Roughly 13 percent of reviews are removed. The loss is balanced across classes,
so it biases the dataset toward shorter reviews rather than toward one
sentiment.

The tokenizer warning about sequences longer than 512 is expected here and can
be ignored. Measuring the true length is the whole point of this cell, so
truncation is switched off on purpose. Nothing is fed to the model at this
stage.

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)

prefixed_texts = [QUERY_PREFIX + text for text in reviews["text"]]
encoded_without_truncation = tokenizer(prefixed_texts, truncation=False)
reviews["token_count"] = [len(ids) for ids in encoded_without_truncation["input_ids"]]

fitting_reviews = reviews[reviews["token_count"] <= MAXIMUM_TOKEN_COUNT].reset_index(
    drop=True
)
dropped_count = len(reviews) - len(fitting_reviews)

print(
    f"Dropped {dropped_count} reviews longer than {MAXIMUM_TOKEN_COUNT} tokens "
    f"({dropped_count / len(reviews):.1%} of the cleaned set)"
)
print(f"Retained {len(fitting_reviews)} reviews")
print(f"Class balance: {fitting_reviews['label'].value_counts().to_dict()}")

/home/user/projects/shapley/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (533 > 512). Running this sequence through the model will result in indexing errors


Dropped 6414 reviews longer than 512 tokens (12.9% of the cleaned set)
Retained 43164 reviews
Class balance: {0: 21643, 1: 21521}


## Take a stratified sample, then split

Encoding all 43,000 fitting reviews takes over 40 minutes on a CPU, which is too
slow to iterate against when the goal is explanation. A stratified sample of
`SAMPLE_SIZE` reviews keeps the class balance and encodes in about two minutes.

Be honest about the tradeoff: an 80/20 split of 2,000 reviews leaves only 400
test reviews, so the accuracy figure below carries roughly plus or minus 3
percent of sampling noise at 95 percent confidence. That is fine for judging
whether the model learned sentiment at all, and it is not a number to quote as a
benchmark result.

In [5]:
from sklearn.model_selection import train_test_split

sampled_reviews, _ = train_test_split(
    fitting_reviews,
    train_size=SAMPLE_SIZE,
    random_state=RANDOM_SEED,
    stratify=fitting_reviews["label"],
)
print(f"Sampled {len(sampled_reviews)} of {len(fitting_reviews)} fitting reviews")
print(f"Class balance: {sampled_reviews['label'].value_counts().to_dict()}")

train_reviews, test_reviews = train_test_split(
    sampled_reviews,
    test_size=TEST_FRACTION,
    random_state=RANDOM_SEED,
    stratify=sampled_reviews["label"],
)
train_reviews = train_reviews.reset_index(drop=True)
test_reviews = test_reviews.reset_index(drop=True)

print(f"Train reviews: {len(train_reviews)}")
print(f"Test reviews: {len(test_reviews)}")

Sampled 2000 of 43164 fitting reviews
Class balance: {0: 1003, 1: 997}
Train reviews: 1600
Test reviews: 400


## Embed with the frozen encoder

e5 pools by averaging token vectors under the attention mask and then L2
normalizes, which is what `1_Pooling/config.json` in the model repository
declares. The encoder is never trained.

Batches are built in length order. Padding is per batch, so mixing a 40 token
review with a 500 token one forces the short one to be computed at 500 tokens.
Sorting by length first keeps padding close to the real token count and is
several times faster on the same hardware. The original row order is restored
before returning, so callers never see the reordering.

In [6]:
import numpy as np
import torch
from transformers import AutoModel

torch.set_num_threads(TORCH_THREAD_COUNT)
encoder = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME).eval()


def embed(texts: list[str]) -> np.ndarray:
    """Encode texts into L2 normalized, mean pooled e5 embeddings.

    The query prefix is applied here rather than by the caller, so every path
    into the encoder is prefixed consistently. Batching follows length order to
    limit padding waste, and the original order is restored before returning.
    """
    length_order = sorted(range(len(texts)), key=lambda index: len(texts[index]))
    batches = []

    for start in range(0, len(length_order), BATCH_SIZE):
        batch_indices = length_order[start : start + BATCH_SIZE]
        tokenized = tokenizer(
            [QUERY_PREFIX + texts[index] for index in batch_indices],
            padding=True,
            truncation=True,
            max_length=MAXIMUM_TOKEN_COUNT,
            return_tensors="pt",
        )
        with torch.inference_mode():
            token_vectors = encoder(**tokenized).last_hidden_state

        mask = tokenized["attention_mask"].unsqueeze(-1)
        pooled = (token_vectors * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        batches.append(torch.nn.functional.normalize(pooled, p=2, dim=1).numpy())

    in_length_order = np.vstack(batches)
    in_original_order = np.empty_like(in_length_order)
    in_original_order[length_order] = in_length_order
    return in_original_order

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18971.42it/s]

In [7]:
TRAIN_EMBEDDINGS_PATH = CHECKPOINT_DIRECTORY / "train_embeddings.npy"
TEST_EMBEDDINGS_PATH = CHECKPOINT_DIRECTORY / "test_embeddings.npy"

if TRAIN_EMBEDDINGS_PATH.exists() and TEST_EMBEDDINGS_PATH.exists():
    train_embeddings = np.load(TRAIN_EMBEDDINGS_PATH)
    test_embeddings = np.load(TEST_EMBEDDINGS_PATH)
    print("Loaded cached embeddings from the checkpoint")
else:
    print("Encoding, expect roughly two minutes")
    train_embeddings = embed(train_reviews["text"].tolist())
    test_embeddings = embed(test_reviews["text"].tolist())
    np.save(TRAIN_EMBEDDINGS_PATH, train_embeddings)
    np.save(TEST_EMBEDDINGS_PATH, test_embeddings)

print(f"Train embeddings: {train_embeddings.shape}")
print(f"Test embeddings: {test_embeddings.shape}")

Loaded cached embeddings from the checkpoint
Train embeddings: (1600, 384)
Test embeddings: (400, 384)


## Train the classification head

A single hidden layer over frozen embeddings. This is the only part that learns,
and it takes seconds.

In [8]:
from sklearn.neural_network import MLPClassifier

head = MLPClassifier(
    hidden_layer_sizes=(128,),
    early_stopping=True,
    random_state=RANDOM_SEED,
)
head.fit(train_embeddings, train_reviews["label"])

print(f"Stopped after {head.n_iter_} iterations")

Stopped after 19 iterations


## Evaluate on the held out split

In [9]:
from sklearn.metrics import accuracy_score, classification_report

predicted_labels = head.predict(test_embeddings)
accuracy = accuracy_score(test_reviews["label"], predicted_labels)

print(f"Held out accuracy: {accuracy:.4f}")
print()
print(
    classification_report(
        test_reviews["label"],
        predicted_labels,
        target_names=["negative", "positive"],
    )
)

Held out accuracy: 0.9150

              precision    recall  f1-score   support

    negative       0.91      0.92      0.92       201
    positive       0.92      0.91      0.91       199

    accuracy                           0.92       400
   macro avg       0.92      0.91      0.91       400
weighted avg       0.92      0.92      0.91       400



## The SHAP contract

`predict_probabilities` is what the next notebook hands to SHAP, and it is the
actual deliverable of this one. Three requirements come from how SHAP calls it:

- SHAP passes a numpy array of strings rather than a list, so both are accepted.
- A fully masked input arrives as an empty or near empty string, which must not
  raise.
- SHAP evaluates hundreds of masked variants per explanation, so batching
  happens inside the function.

In [10]:
def predict_probabilities(texts) -> np.ndarray:
    """Map raw review texts to class probabilities of shape (n, 2).

    Accepts any sequence of strings, including the numpy arrays that SHAP
    passes. The e5 query prefix is applied inside `embed`, which keeps the fixed
    prefix out of reach of the SHAP masker.
    """
    return head.predict_proba(embed([str(text) for text in texts]))


sample_reviews = test_reviews["text"].head(3).tolist()
print("Probabilities for three test reviews:")
print(predict_probabilities(np.array(sample_reviews)))
print()
print("True labels:", test_reviews["label"].head(3).tolist())
print("Empty input is handled:", predict_probabilities([""]))

Probabilities for three test reviews:


[[0.4691202  0.5308798 ]
 [0.39306444 0.60693556]
 [0.24183208 0.7581679 ]]

True labels: [1, 1, 1]
Empty input is handled: [[0.46792692 0.5320731 ]]


## SHAP smoke test

Rather than assume the interface works, verify the local accuracy property:
attributions summed over all tokens, plus the base value, must reproduce the
model output for that input. If this passes, the next notebook can start on
explanation instead of debugging plumbing.

In [11]:
import shap

shortest_review = test_reviews.nsmallest(1, "token_count")["text"].iloc[0]
print(f"Explaining a {len(shortest_review.split())} word review:")
print(shortest_review)
print()

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(predict_probabilities, masker)
explanation = explainer([shortest_review])

reconstructed = explanation.base_values[0] + explanation.values[0].sum(axis=0)
actual = predict_probabilities([shortest_review])[0]

print(f"Base value plus attributions: {reconstructed}")
print(f"Actual model output:          {actual}")

np.testing.assert_allclose(reconstructed, actual, atol=0.01)
print()
print("SHAP local accuracy holds. predict_probabilities is explainer ready.")

Explaining a 27 word review:
I really enjoyed this movie... In My DVD collection of baseball movies... Reminded me how great the sport truly is... Whether it's here in America or Japan.



Base value plus attributions: [0.13900882 0.86099118]
Actual model output:          [0.13900882 0.8609912 ]

SHAP local accuracy holds. predict_probabilities is explainer ready.


## Save the checkpoint

Everything the SHAP notebook needs, written under the sample specific checkpoint
directory. The metadata carries only what is required to rebuild
`predict_probabilities`, plus the sample size so a checkpoint can never be
confused with one built from a different subset.

In [12]:
import json

import joblib

joblib.dump(head, CHECKPOINT_DIRECTORY / "head.joblib")
train_reviews.to_parquet(CHECKPOINT_DIRECTORY / "train.parquet", index=False)
test_reviews.to_parquet(CHECKPOINT_DIRECTORY / "test.parquet", index=False)

metadata = {
    "embedding_model_name": EMBEDDING_MODEL_NAME,
    "maximum_token_count": MAXIMUM_TOKEN_COUNT,
    "query_prefix": QUERY_PREFIX,
    "random_seed": RANDOM_SEED,
    "sample_size": SAMPLE_SIZE,
    "train_size": len(train_reviews),
    "test_size": len(test_reviews),
    "test_accuracy": float(accuracy),
}
(CHECKPOINT_DIRECTORY / "metadata.json").write_text(json.dumps(metadata, indent=2))

print(json.dumps(metadata, indent=2))
print()
print("Checkpoint contents:", sorted(path.name for path in CHECKPOINT_DIRECTORY.iterdir()))

{
  "embedding_model_name": "intfloat/e5-small-v2",
  "maximum_token_count": 512,
  "query_prefix": "query: ",
  "random_seed": 42,
  "sample_size": 2000,
  "train_size": 1600,
  "test_size": 400,
  "test_accuracy": 0.915
}

Checkpoint contents: ['head.joblib', 'metadata.json', 'test.parquet', 'test_embeddings.npy', 'train.parquet', 'train_embeddings.npy']
